# Agentic RAG — From First Principles

Standard RAG performs one retrieval pass and one generation pass. Agentic RAG relaxes that constraint. Instead of assuming the first search is enough, we let the system think in steps: retrieve, inspect, compare, summarize, calculate, and retry if needed.

The core idea is not "agents" as marketing. The core idea is **explicit tool use**. An agent is simply a loop that chooses which tool to call next based on what it has already observed.

## Why Single-Turn RAG Is Not Enough

Some questions require more than one retrieval action:

- comparisons across multiple entities,
- numerical reasoning over retrieved facts,
- synthesis across dispersed evidence,
- recovery when the first retrieval misses the target.

That is where an agent loop becomes useful. The loop turns retrieval from a one-shot helper into a reusable tool.

## A Minimal ReAct-Style Loop

```
observe question
  -> choose tool
  -> execute tool
  -> inspect result
  -> choose next tool or finish
```

The loop needs only a few ingredients: a tool registry, a planner, a memory trace, and guard rails.

## The ReAct Framework Deep Dive

The **ReAct** paradigm (Yao et al., 2022) unifies **Rea**soning and **Act**ing in a
single interleaved trace. Unlike pure chain-of-thought (CoT), which reasons
internally before producing a final answer, ReAct alternates between three
explicit stages:

| Stage | What happens | Example |
|---|---|---|
| **Thought** | The model reasons about what it knows and what it still needs. | *"I know HelioGrid added storage, but I need curtailment data."* |
| **Action** | The model selects a tool and an argument. | `retrieve("HelioGrid curtailment rate")` |
| **Observation** | The environment returns the tool's output. | *"Curtailment dropped from 12 % to 4 %."* |

### State-Machine View

```
            ┌──────────┐
            │  START    │
            └────┬─────┘
                 │ receive question
                 ▼
          ┌─────────────┐
     ┌───▶│  THOUGHT    │◀────────────────┐
     │    └──────┬──────┘                  │
     │           │ choose tool             │
     │           ▼                         │
     │    ┌─────────────┐                  │
     │    │   ACTION    │                  │
     │    └──────┬──────┘                  │
     │           │ execute tool            │
     │           ▼                         │
     │    ┌──────────────┐   need more     │
     │    │ OBSERVATION  │────info──────────┘
     │    └──────┬───────┘
     │           │ sufficient
     │           ▼
     │    ┌─────────────┐
     └────│   ANSWER    │
          └─────────────┘
```

### How ReAct Differs from Chain-of-Thought

| Dimension | Chain-of-Thought | ReAct |
|---|---|---|
| External tools | None — reasoning is purely internal | Tools provide fresh evidence each step |
| Grounding | Can hallucinate unsupported facts | Observations ground each reasoning step |
| Transparency | One long internal monologue | Explicit thought/action/observation trace |
| Error correction | Must get reasoning right in one pass | Can re-plan after unexpected observations |

### Key Contributions of Yao et al. (2022)

1. **Interleaved trace format** — thoughts and actions appear in the *same*
   token stream, so a single auto-regressive model handles both.
2. **Grounded reasoning** — each thought is preceded by a real observation,
   reducing hallucination on knowledge-intensive tasks.
3. **Generality** — the same loop works for QA, fact-checking, and
   interactive decision-making with only tool-set changes.

Our implementation below follows this pattern: the `plan_step` function
produces the *thought + action*, and the agent loop executes the action
and records the *observation* before the next planning call.

## Environment Setup

All notebooks in this overhaul use the same minimal native stack:

- **Qwen/Qwen3-14B** for generation
- **BAAI/bge-base-en-v1.5** for embeddings
- **FAISS** for similarity search
- **Google Drive** for persistent Colab caching

The goal is to keep the pipeline transparent. Every major component is visible as plain Python functions instead of framework abstractions.

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy
print("Installed core native RAG dependencies.")

In [ ]:
import time
import math
import json
import re
from collections import defaultdict, Counter

import numpy as np
import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount("/content/drive")
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto"
)
print("Loaded", MODEL_NAME)

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)
print("Loaded embedding model:", embed_model.__class__.__name__)

In [ ]:
def generate(prompt, max_new_tokens=220, temperature=0.2):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        top_k=20,
        temperature=temperature,
        do_sample=temperature > 0,
    )
    answer_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(answer_ids, skip_special_tokens=True).strip()

print("Generation helper ready.")

## Knowledge Base for Multi-Step Questions

We use a small fictional clean-energy ecosystem so the agent has to retrieve, compare, and occasionally calculate.

In [ ]:
kb = [
    {"id": "c1", "entity": "HelioGrid", "text": "HelioGrid runs solar farms and added 240 MWh of battery storage in 2022. Its reported curtailment fell from 14% to 8% after the expansion."},
    {"id": "c2", "entity": "Northwind", "text": "Northwind operates offshore turbines with strong nighttime output. It signed a balancing agreement with HelioGrid to reduce evening net-load ramps."},
    {"id": "c3", "entity": "VoltShift", "text": "VoltShift is a demand-response aggregator serving industrial refrigeration and electric vehicle fleets. It can shed or shift load within minutes."},
    {"id": "c4", "entity": "BlueHydra", "text": "BlueHydra develops hydrogen storage for long-duration backup. The system reacts slowly but can supply energy for much longer than lithium-ion batteries."},
    {"id": "c5", "entity": "Regulation", "text": "The 2025 Flex Grid Act rewards resources that perform during evening peaks and rapid ramps. Batteries and demand response often score well because they respond quickly."},
    {"id": "c6", "entity": "Metrics", "text": "Analysts compare flexibility resources using response speed, duration, capital cost, utilization rate, and dependence on transmission."},
]

print("Knowledge base entries:", len(kb))
for row in kb:
    print(row["id"], "-", row["entity"])

In [ ]:
kb_texts = [row["text"] for row in kb]
kb_embeddings = embed_model.encode(kb_texts, normalize_embeddings=True)
kb_embeddings = np.asarray(kb_embeddings, dtype="float32")
kb_index = faiss.IndexFlatIP(kb_embeddings.shape[1])
kb_index.add(kb_embeddings)
print("Vector index ready:", kb_index.ntotal)

## Defining Tools as Plain Python Functions

Frameworks often hide tool execution behind decorators and orchestration objects. We do not need any of that. A tool is just a function with a documented contract.

In [ ]:
def retrieve_tool(query, k=3):
    vec = embed_model.encode([query], normalize_embeddings=True)
    vec = np.asarray(vec, dtype="float32")
    scores, indices = kb_index.search(vec, k)
    rows = []
    for score, idx in zip(scores[0], indices[0]):
        rows.append({
            "id": kb[idx]["id"],
            "entity": kb[idx]["entity"],
            "score": float(score),
            "text": kb[idx]["text"],
        })
    return rows

def compare_tool(left, right):
    left_hits = retrieve_tool(left, k=2)
    right_hits = retrieve_tool(right, k=2)
    return {"left": left_hits, "right": right_hits}

def summarize_tool(text):
    return generate(f"Summarize this evidence in 2 bullet-style sentences.\n\n{text}", max_new_tokens=100)

def calculator_tool(expression):
    allowed = re.sub(r"[^0-9\.\+\-\*\/\(\) ]", "", expression)
    return eval(allowed)

print("retrieve_tool sample:", retrieve_tool("battery expansion at HelioGrid", k=1))

In [ ]:
TOOLS = {
    "retrieve": retrieve_tool,
    "compare": compare_tool,
    "summarize": summarize_tool,
    "calculate": calculator_tool,
    "finish": None,
}

for name in TOOLS:
    print("Tool:", name)

## Planning the Next Action

An agent loop needs a planner. In production you might use structured tool-calling APIs. Here we keep the planner transparent: we ask the model for a small JSON object and then parse it carefully with fallbacks.

In [ ]:
def plan_step(question, scratchpad):
    prompt = f"""
You are a retrieval agent. Choose the next tool.
Return JSON with keys: action, argument.
Allowed actions: retrieve, compare, summarize, calculate, finish.

Question: {question}
Scratchpad:
{scratchpad}
""".strip()

    raw = generate(prompt, max_new_tokens=120)
    match = re.search(r"\{.*\}", raw, flags=re.S)
    if match:
        try:
            data = json.loads(match.group(0))
            if data.get("action") in TOOLS:
                return data
        except json.JSONDecodeError:
            pass

    q = question.lower()
    if "compare" in q or "versus" in q or "vs" in q:
        return {"action": "compare", "argument": question}
    if any(token in q for token in ["how many", "difference", "drop", "percent"]):
        return {"action": "calculate", "argument": "14 - 8"}
    return {"action": "retrieve", "argument": question}

print(plan_step("Compare batteries versus hydrogen storage.", ""))

## The Agent Loop

The loop below is intentionally small. That is the point. Agentic behavior does not require huge orchestration code. It requires a sequence of explicit decisions, tool executions, and guard rails.

In [ ]:
def run_agent(question, max_steps=4):
    scratchpad = []
    seen_actions = set()

    for step in range(max_steps):
        plan = plan_step(question, "\n".join(scratchpad))
        action = plan["action"]

        if action == "finish":
            break

        if (action, str(plan.get("argument"))) in seen_actions and action != "summarize":
            scratchpad.append("Repeated action detected; switching to finish.")
            break
        seen_actions.add((action, str(plan.get("argument"))))

        if action == "retrieve":
            result = retrieve_tool(plan["argument"], k=3)
            scratchpad.append(f"STEP {step+1}: retrieve -> {result}")
        elif action == "compare":
            text = question.lower().replace("compare", "").strip().replace("versus", "|").replace("vs", "|")
            parts = [part.strip(" ?.") for part in text.split("|") if part.strip()]
            if len(parts) < 2:
                result = retrieve_tool(question, k=3)
                scratchpad.append(f"STEP {step+1}: fallback retrieve -> {result}")
            else:
                result = compare_tool(parts[0], parts[1])
                scratchpad.append(f"STEP {step+1}: compare -> {result}")
        elif action == "calculate":
            try:
                value = calculator_tool(plan["argument"])
                scratchpad.append(f"STEP {step+1}: calculate -> {value}")
            except Exception as exc:
                scratchpad.append(f"STEP {step+1}: calculate failed -> {exc}")
        elif action == "summarize":
            result = summarize_tool("\n".join(scratchpad))
            scratchpad.append(f"STEP {step+1}: summarize -> {result}")

        if len(scratchpad) >= 2:
            scratchpad.append("STEP final: finish")
            break

    final_prompt = f"""
Answer the question using the trace below. Be explicit when evidence is limited.

Question: {question}
Trace:
{chr(10).join(scratchpad)}
""".strip()

    return {"trace": scratchpad, "answer": generate(final_prompt, max_new_tokens=180)}

demo = run_agent("How did HelioGrid's storage expansion affect curtailment?")
print("\nTRACE:")
for line in demo["trace"]:
    print(line)
print("\nANSWER:\n", demo["answer"])

## Trace Inspection and Debugging

When an agent produces an unexpected answer, the **trace** is your primary
debugging tool. Each entry records the thought, the action chosen, and the
observation returned by the environment. By inspecting the full trace you can
pinpoint:

- **Good decisions** — the agent retrieved the right evidence at the right step.
- **Suboptimal decisions** — a retrieval was too broad, or the agent
  summarised too early before gathering enough facts.
- **Failure modes** — the planner hallucinated a tool name, repeated the same
  query, or hit the step limit without producing an answer.

Below we run a deliberately complex query and annotate each step.

In [ ]:
# Run a complex multi-step query and inspect every step of the trace
complex_q = (
    "Which company stores more energy — HelioGrid with batteries or "
    "AquaVolt with pumped hydro — and how does that affect grid stability?"
)
result = run_agent(complex_q, max_steps=5)

print("═" * 70)
print(f"Question: {complex_q}")
print("═" * 70)

for i, entry in enumerate(result["trace"]):
    step_num = i + 1
    # Classify each step for debugging
    if "retrieve" in entry.lower():
        label = "✓ RETRIEVAL"
    elif "compare" in entry.lower():
        label = "✓ COMPARISON"
    elif "summarize" in entry.lower() or "answer" in entry.lower():
        label = "→ SYNTHESIS"
    else:
        label = "? UNKNOWN"
    print(f"\n--- Step {step_num} [{label}] ---")
    print(entry)

print("\n" + "═" * 70)
print("Final answer:", result["answer"])
print("═" * 70)

# Debugging checklist
trace_text = " ".join(result["trace"]).lower()
checks = [
    ("heliogrid" in trace_text, "Retrieved HelioGrid info"),
    ("aquavolt" in trace_text, "Retrieved AquaVolt info"),
    ("compare" in trace_text, "Used comparison tool"),
    (len(result["trace"]) <= 5, f"Completed in {len(result['trace'])} steps (≤5)"),
]
print("\nDebug checklist:")
for passed, desc in checks:
    icon = "✅" if passed else "❌"
    print(f"  {icon} {desc}")

## Multi-Step Examples

The value of an agent loop becomes clearer on tasks that naturally require more than one tool.

In [ ]:
examples = [
    "How did HelioGrid's storage expansion affect curtailment?",
    "Compare batteries versus hydrogen storage for peak support.",
    "What is the drop in curtailment from 14% to 8%?",
]

for q in examples:
    print("\nQUESTION:", q)
    result = run_agent(q)
    print("TRACE LENGTH:", len(result["trace"]))
    print("ANSWER:", result["answer"][:220])

## Error Recovery and Unknown Tools

Agents are only useful if they fail gracefully. If the planner emits an unknown action, repeats itself, or produces malformed output, the loop should degrade into a safe retrieval path rather than crash.

In [ ]:
bad_plan = {"action": "teleport", "argument": "none"}
safe_action = bad_plan["action"] if bad_plan["action"] in TOOLS else "retrieve"
print("Recovered action:", safe_action)
print("Recovered result:", retrieve_tool("Which resources respond quickly during evening peaks?", k=2))

## Guard Rails

The two most common failure modes in lightweight agents are:

1. infinite loops,
2. repeated low-value actions.

Both are mitigated here with a step limit and duplicate-action detection. In production you might also add budget limits, confidence thresholds, and explicit "stop if no new evidence" checks.

In [ ]:
guard_demo = run_agent("Compare batteries versus hydrogen storage for peak support.", max_steps=3)
print("Guarded trace:")
for step in guard_demo["trace"]:
    print(" -", step)

## Baseline RAG vs Agentic RAG

Standard RAG can answer many questions well. Agentic RAG is not automatically better. It is better when the question requires multiple retrieval actions, explicit comparison, or simple computation over retrieved evidence.

In [ ]:
def baseline_answer(question):
    hits = retrieve_tool(question, k=3)
    context = "\n\n".join([f"[{row['id']}] {row['text']}" for row in hits])
    return generate(f"Use this evidence only.\n\n{context}\n\nQuestion: {question}\n\nAnswer:", max_new_tokens=150)

question = "Compare batteries versus hydrogen storage for peak support."
print("BASELINE:\n", baseline_answer(question))
print("\nAGENTIC:\n", run_agent(question)["answer"])

## Query Complexity Analysis

Not every query benefits from the agent loop. We can classify questions into
four complexity tiers and match each to the best retrieval strategy:

| Tier | Description | Example | Best approach |
|---|---|---|---|
| **Simple factual** | Single fact, one retrieval needed | *"When did HelioGrid add storage?"* | Baseline RAG |
| **Multi-hop** | Answer requires 2+ retrievals chained together | *"How did HelioGrid's storage affect curtailment?"* | Agentic RAG |
| **Comparative** | Must gather facts about 2+ entities and contrast | *"Compare battery vs hydrogen storage"* | Agentic RAG |
| **Analytical** | Requires calculation or aggregation over retrieved data | *"What is the combined capacity of all storage?"* | Agentic RAG + calculate tool |

### Decision Matrix

```
 Query needs only 1 retrieval?  ──YES──▶  Use baseline RAG  (faster, cheaper)
         │
        NO
         │
 Query compares entities?  ──YES──▶  Use agentic RAG with compare tool
         │
        NO
         │
 Query needs calculation?  ──YES──▶  Use agentic RAG with calculate tool
         │
        NO
         │
 Multi-hop reasoning needed?  ──YES──▶  Use agentic RAG (retrieve → retrieve → summarize)
         │
        NO
         ▼
 Default to baseline RAG
```

Below we classify the example queries and verify that the decision matrix
matches empirical quality.

In [ ]:
# Classify queries by complexity and compare baseline vs agentic performance
test_queries = [
    {"q": "When did HelioGrid add battery storage?",
     "tier": "simple_factual", "expected_best": "baseline"},
    {"q": "How did HelioGrid's storage expansion affect curtailment?",
     "tier": "multi_hop", "expected_best": "agentic"},
    {"q": "Compare batteries versus hydrogen storage for peak support.",
     "tier": "comparative", "expected_best": "agentic"},
    {"q": "What is the combined storage capacity of HelioGrid and AquaVolt?",
     "tier": "analytical", "expected_best": "agentic"},
]

print(f"{'Tier':<18} {'Tokens (Base)':>14} {'Tokens (Agent)':>14} {'Recommended':>12}")
print("─" * 62)

for tq in test_queries:
    # Baseline
    b_start = time.time()
    b_ans = baseline_answer(tq["q"])
    b_time = time.time() - b_start
    b_tokens = len(b_ans.split())

    # Agentic
    a_start = time.time()
    a_result = run_agent(tq["q"])
    a_time = time.time() - a_start
    a_tokens = len(a_result["answer"].split())
    a_steps = len(a_result["trace"])

    recommend = "agentic" if a_steps > 1 else "baseline"
    match = "✅" if recommend == tq["expected_best"] else "⚠️"
    print(f"{tq['tier']:<18} {b_tokens:>10} tok {a_tokens:>10} tok {recommend:>10} {match}")

print("\nKey: ✅ = matches expected, ⚠️ = mismatch (review query classification)")

## A Reusable AgenticRAG Class

At this point we can package the loop without hiding its internals. Encapsulation is useful as long as the behavior remains inspectable.

In [ ]:
class AgenticRAG:
    def ask(self, question):
        return run_agent(question)

agent = AgenticRAG()
response = agent.ask("How do batteries, offshore wind, and demand response complement each other?")
print("Trace steps:", len(response["trace"]))
print(response["answer"])

## Cost, Latency, and When to Use Agentic RAG

Agentic RAG adds planning overhead. It usually means more model calls and more branching logic. That cost is justified when the question is genuinely multi-step. It is not justified when a single retrieval pass already answers the question.

So the design question is not *"Can I build an agent?"* but *"Which queries deserve an agent loop?"* In many systems the right answer is hybrid: baseline RAG for simple questions, agentic RAG for hard cases.

In [ ]:
start = time.time()
for q in examples:
    _ = run_agent(q)
elapsed = time.time() - start
print(f"Processed {len(examples)} agentic queries in {elapsed:.2f}s")
print(f"Average time per query: {elapsed / len(examples):.2f}s")

## Scaling Considerations

Moving from a notebook demo to a production system introduces constraints
that do not appear at small scale.

### Token Cost per Agent Loop

Each planning call sends the **full scratchpad** as context. If the model
generates ~200 tokens per step, a 4-step trace means:

```
Step 1:  prompt_base + 0 history tokens
Step 2:  prompt_base + ~200 history tokens
Step 3:  prompt_base + ~400 history tokens
Step 4:  prompt_base + ~600 history tokens
Total ≈  4 × prompt_base + 1 200 history tokens
```

Costs grow **quadratically** with the number of steps because each step
re-reads all previous steps.

### Latency Implications

- Each step requires a **full model forward pass** plus tool execution.
- With a 4-bit quantised 14 B model on a T4 GPU, expect ~1-3 s per step.
- A 4-step agent call therefore takes 4-12 s — acceptable for research,
  slow for interactive products.

### Production Patterns

| Pattern | Purpose |
|---|---|
| **Max-step cap** | Prevent runaway loops; we already use `max_steps`. |
| **Timeout** | Hard wall-clock limit (e.g. 30 s) to avoid tail latency. |
| **Fallback to baseline** | If the agent exceeds *N* steps, return a single-retrieval answer instead. |
| **Scratchpad compression** | Summarise older observations to keep the prompt short. |
| **Caching** | Cache tool outputs — the same retrieval query often recurs. |

### When to Use Simpler Approaches

- **> 80 % of queries are single-hop** → baseline RAG with an optional
  agent fallback is cheaper.
- **Strict latency SLA (< 2 s)** → agentic overhead is too high; use
  pre-computed multi-hop indices instead.
- **Cost-sensitive batch jobs** → run baseline first, route only low-confidence
  answers to the agent.

In [ ]:
# Estimate token cost growth as the number of agent steps increases
prompt_base_tokens = 350  # approximate base prompt size
tokens_per_step = 200     # approximate tokens added per step

print(f"{'Steps':>6} {'Total prompt tokens':>22} {'Growth vs step 1':>18}")
print("─" * 50)

step1_cost = None
for n_steps in range(1, 9):
    # Each step i reads prompt_base + all previous step outputs
    total = sum(prompt_base_tokens + i * tokens_per_step for i in range(n_steps))
    if step1_cost is None:
        step1_cost = total
    ratio = total / step1_cost
    print(f"{n_steps:>6} {total:>18} tok {ratio:>14.1f}x")

print(f"\nTakeaway: 8 steps costs ~{8*7//2 + 8:.0f}x the base prompt in history tokens alone.")
print("Keep max_steps low (3-5) and compress the scratchpad for production use.")

## Key Takeaways

Agentic RAG is best understood as **tool-using retrieval** rather than as a mysterious autonomous system. Once you define tools, traces, and guard rails clearly, the agent loop becomes an understandable extension of ordinary RAG rather than a separate discipline.

In [ ]:
tool_use_counts = Counter()
for q in examples:
    for line in run_agent(q)["trace"]:
        if ":" in line:
            action_name = line.split(":")[1].split("->")[0].strip().split()[0]
            tool_use_counts[action_name] += 1

print(dict(tool_use_counts))